In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Implementing RNNs

that section wrote down a recurrent language model and counted its
parameters, but did not train it. Two questions are open: how its
equations become working code, and what its hidden state is worth on *The
Time Machine*, measured against the counts of
that section. The first is the easier one. The model fits in three equations: look up the
embedding of the current token, update the hidden state from it and from
the previous state, and map the new state to scores over the vocabulary.
The code for them has a simple interface. At each time step the model
consumes one token per sequence and the previous hidden state, and it
returns $|\mathcal{V}|$ logits per sequence and a new state of the same
shape as the old one. Because the shapes do not change from step to step,
the same step applies to a sequence of any length.

Let us first write this step with plain tensor operations, so that every
parameter and every shape is visible, and then hand it to the recurrent
layer that each framework provides. Training needs one addition that the
equations do not show. The gradient passes through the same transition once
per time step and can become large enough to undo much of the progress of
training in a single update. We therefore clip its norm before every
update, a crude remedy and a standard one, and we train once without it to
see what it prevents. The second question we answer by measurement. The
trained model is scored against the count-based $n$-gram models on the same
text, token for token, and then position by position, to find out how much
of what it has read the model uses. Finally we let it generate text.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

## An RNN Language Model from Scratch

The model has three parts, and we implement each as its own class or method:
a *recurrent cell* that updates the hidden state one time step at a time, an
embedding that turns token ids into the vectors the cell consumes, and an
output layer that maps each hidden state to a vector of logits over the
vocabulary.

### The Recurrent Cell

For a minibatch of $b$ sequences, let $\mathbf{X}_t \in \mathbb{R}^{b
\times d}$ hold their inputs at step $t$ and $\mathbf{H}_t \in
\mathbb{R}^{b \times h}$ their hidden states. The cell computes the
recurrence of that section with the tanh activation,

$$\mathbf{H}_t = \tanh(\mathbf{X}_t \mathbf{W}_{\textrm{xh}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hh}} + \mathbf{b}_\textrm{h}),$$

so each new state mixes the current input with the previous state through
two weight matrices, $\mathbf{W}_{\textrm{xh}} \in \mathbb{R}^{d \times
h}$ and $\mathbf{W}_{\textrm{hh}} \in \mathbb{R}^{h \times h}$, and a
bias $\mathbf{b}_\textrm{h} \in \mathbb{R}^h$. The `RNNScratch` class
holds these three parameters. The number of hidden units `num_hiddens` ($h$)
is a hyperparameter, and `num_inputs` ($d$) is the width of the vectors
arriving at each step, which will shortly be the embedding dimension.

As in that section, we register the weights with the framework's
parameter type, so that the optimizer finds them and the trainer can move
them to a GPU. Both weight matrices start from $\mathcal{N}(0, \sigma^2)$
with $\sigma = 0.01$, and the bias starts at zero. With weights this
small, the largest eigenvalue of the recurrent weight matrix has magnitude
about $\sigma\sqrt{h} \approx 0.11$. Recall the scalar example of
that section, where the past persisted scaled by about $w_\textrm{hh}$
per step: at 0.11 per step, the untrained cell forgets its state within a
few steps. Training has to grow the weights from there.

Each weight gets its own initializer object. An unseeded Keras initializer
returns the same values every time it is called, so one shared initializer
would make the first rows of $\mathbf{W}_{\textrm{hh}}$ a copy of
$\mathbf{W}_{\textrm{xh}}$.

In [ ]:
class RNNScratch(d2l.Module):
    """The RNN model implemented from scratch."""
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()
        self.W_xh = self.add_weight(  # One initializer per weight
            name='W_xh', shape=(num_inputs, num_hiddens),
            initializer=tf.keras.initializers.RandomNormal(stddev=sigma))
        self.W_hh = self.add_weight(
            name='W_hh', shape=(num_hiddens, num_hiddens),
            initializer=tf.keras.initializers.RandomNormal(stddev=sigma))
        self.b_h = self.add_weight(name='b_h', shape=(num_hiddens,),
                                   initializer='zeros')

The forward computation loops over the outermost axis of `inputs`, whose
shape is (`num_steps`, `batch_size`, `num_inputs`), applies the recurrence
once per time step, and collects the hidden state of every step. When no
initial state is supplied, the state starts at zero. The explicit Python
loop over time steps is the recurrence as we wrote it on paper, which is
all we ask of it for now.

In [ ]:
@d2l.add_to_class(RNNScratch)
def forward(self, inputs, state=None):
    if state is None:
        # Initial state with shape: (batch_size, num_hiddens)
        state = tf.zeros((tf.shape(inputs)[1], self.num_hiddens))
    outputs = []
    for X in tf.unstack(inputs):  # Shape: (num_steps, batch_size, num_inputs)
        state = d2l.tanh(d2l.matmul(X, self.W_xh) +
                         d2l.matmul(state, self.W_hh) + self.b_h)
        outputs.append(state)
    return outputs, state

We feed the cell a minibatch of two sequences of 100 steps each.

In [ ]:
batch_size, num_inputs, num_hiddens, num_steps = 2, 16, 32, 100
rnn = RNNScratch(num_inputs, num_hiddens)
X = d2l.ones((num_steps, batch_size, num_inputs))
outputs, state = rnn(X)

We expect one state per step, each with the shape (`batch_size`,
`num_hiddens`) of the state the cell started from. The helper below checks
a shape; later sections use it as well.

In [ ]:
def check_shape(a, shape):
    """Check the shape of a tensor."""
    assert a.shape == shape, \
            f'tensor\'s shape {a.shape} != expected shape {shape}'

check_shape(d2l.stack(outputs, 0), (num_steps, batch_size, num_hiddens))
check_shape(state, (batch_size, num_hiddens))

### The Embedding Table

The cell consumes vectors, but our data pipeline delivers integer token ids.
As in that section, an embedding lookup bridges the gap: a trainable
table $\mathbf{E} \in \mathbb{R}^{|\mathcal{V}| \times d}$ represents
token $i$ by its $i^\textrm{th}$ row, a $d$-dimensional vector that
gradient descent adjusts along with every other parameter. Older treatments
used a *one-hot encoding* instead: token $i$ becomes the
length-$|\mathcal{V}|$ vector with a single 1 in position $i$, and the
input weights multiply that vector. Since multiplying a one-hot vector by a
matrix selects a row of it (that section), this is nothing but a
lookup, computed the expensive way. With a vocabulary of 27 characters
nobody minds. With our BPE vocabulary of 1,027 ids, each step would multiply
a vector that is zero everywhere but in one entry.

The `RNNLMScratch` class assembles the language model around a cell `rnn`:
the embedding table `E` on the input side and an output layer (`W_hq`,
`b_q`) on the output side. The embedding dimension is read off the cell's
`num_inputs`, since the cell consumes what the embedding produces. The
constructor also takes two dropout rates. Both are zero for now, and we
return to them when the model overfits. The embedding rows start at unit
scale, as the 1s of a one-hot encoding would, while the output weights
start at scale $\sigma$ and the output bias at zero. The logits of the
untrained model are then close to zero, its next-token distribution is
close to uniform, and its perplexity is close to $|\mathcal{V}|$. This is
a prediction we can check before training.

The model inherits from `Classifier` the cross-entropy
loss, averaged over all tokens of a minibatch. Its validation step returns
only this loss, since the token accuracy that `Classifier` would add is not
what we monitor. The mean cross-entropy is the logarithm of the perplexity
(that section), so the validation perplexity is the
exponential of the trainer's mean validation loss.

In [ ]:
class RNNLMScratch(d2l.Classifier):
    """The RNN-based language model implemented from scratch."""
    def __init__(self, rnn, vocab_size, lr=0.01, dropout=0, emb_dropout=0):
        super().__init__()
        self.save_hyperparameters()
        self.sigma = getattr(rnn, 'sigma', 0.01)  # Scale of the output weights
        self.init_params()

    def init_params(self):
        self.E = self.add_weight(
            name='E', shape=(self.vocab_size, self.rnn.num_inputs),
            initializer=tf.keras.initializers.RandomNormal(stddev=1))
        self.W_hq = self.add_weight(
            name='W_hq', shape=(self.rnn.num_hiddens, self.vocab_size),
            initializer=tf.keras.initializers.RandomNormal(stddev=self.sigma))
        self.b_q = self.add_weight(name='b_q', shape=(self.vocab_size,),
                                   initializer='zeros')

    def validation_step(self, batch):
        return self.loss(self(*batch[:-1], training=False), batch[-1])

The minibatches sampled by our data pipeline have shape (`batch_size`,
`num_steps`). The `embedding` method looks up each id and transposes the
result to (`num_steps`, `batch_size`, `num_inputs`), the layout whose
outermost axis the cell's forward loop walks over.

In [ ]:
@d2l.add_to_class(RNNLMScratch)
def embedding(self, X):
    # Output shape: (num_steps, batch_size, num_inputs)
    return tf.gather(self.E, tf.transpose(X))

### The Output Layer

The last of the three equations turns each hidden state into scores for
the next token. The output layer computes $\mathbf{O}_t = \mathbf{H}_t
\mathbf{W}_{\textrm{hq}} + \mathbf{b}_\textrm{q}$ for every time step, with
$\mathbf{W}_{\textrm{hq}} \in \mathbb{R}^{h \times |\mathcal{V}|}$. Since
the same weights serve every step, one matrix product handles all of them:
the method stacks the states of all steps into a (`batch_size`,
`num_steps`, `num_hiddens`) tensor and multiplies it by
$\mathbf{W}_{\textrm{hq}}$, which gives logits of shape (`batch_size`,
`num_steps`, `vocab_size`) that line up with the target array. The
`forward` method chains the three parts together.

In [ ]:
@d2l.add_to_class(RNNLMScratch)
def output_layer(self, rnn_outputs):
    # A list of per-step states, or a tensor whose first axis is time
    H = d2l.stack(list(rnn_outputs), 1)  # (batch_size, num_steps, num_hiddens)
    return d2l.matmul(H, self.W_hq) + self.b_q

@d2l.add_to_class(RNNLMScratch)
def forward(self, X, state=None):
    embs = self.embedding(X)
    if self.training:  # tf.nn.dropout is always active; apply it in training
        embs = tf.nn.dropout(embs, self.emb_dropout)
    rnn_outputs, _ = self.rnn(embs, state)
    if self.training:  # A list of per-step states (scratch) or one tensor
        rnn_outputs = tf.nest.map_structure(
            lambda H: tf.nn.dropout(H, self.dropout), rnn_outputs)
    return self.output_layer(rnn_outputs)

A minibatch of token ids of shape (`batch_size`, `num_steps`) should come
out as one vector of logits per position. We check this with a vocabulary
of 1,024 ids, about the size of the one we train on.

In [ ]:
model = RNNLMScratch(rnn, vocab_size=1024)
outputs = model(d2l.ones((batch_size, num_steps), dtype=d2l.int64))
check_shape(outputs, (batch_size, num_steps, 1024))

## Gradient Clipping

A recurrent network is deep in time. Backpropagating through $T$ time
steps multiplies $T$ Jacobians of the same transition, one per step, and
as in a deep feedforward network such a product can shrink or grow
exponentially with its length. that section analyzes both cases.
Vanishing gradients limit what the model can learn, but they do not break
training. Exploding gradients can: when the product grows, an occasional
gradient is many times longer than the typical one. Below, we train the
same model with a remedy for them and without it.

To see what a large gradient threatens, consider the length of one update.
A gradient step with learning rate $\eta$ moves the parameters
$\mathbf{w}$ to $\mathbf{w} - \eta\mathbf{g}$, a distance

$$\|(\mathbf{w} - \eta\mathbf{g}) - \mathbf{w}\| = \eta\|\mathbf{g}\|.$$

Thus the length of a step grows with the norm of the gradient. Gradient
descent works because, over a short step, the loss is close to its linear
approximation $f(\mathbf{w}) - \eta\|\mathbf{g}\|^2$, which decreases.
When $\|\mathbf{g}\|$ spikes, the step leaves the region where that
approximation holds, and the loss can jump instead of falling. One such
step can undo the progress of many updates. What can we do about it?
Shrinking $\eta$ until even the rare spike is safe would slow every step to
protect against a few. *Gradient clipping* leaves $\eta$ alone and
instead caps the norm on the occasions it is too large,
projecting $\mathbf{g}$ onto the ball of radius $\theta$:

$$\mathbf{g} \leftarrow \min\left(1, \frac{\theta}{\|\mathbf{g}\|}\right) \mathbf{g}.$$

A gradient with norm at most $\theta$ passes unchanged; a longer one is
scaled down to norm $\theta$ and keeps its direction. With $\theta = 1$, a
gradient of norm 10 becomes one tenth of itself. The norm
$\|\mathbf{g}\|$ is the Euclidean norm of the gradients of all parameters
of the model, stacked into one vector.

Note that this is a rather crude device. Whenever it acts, clipping changes
the update, so training no longer follows the gradient, and its effect on
convergence is hard to analyze. It is standard nevertheless: recurrent
networks and large transformers alike are trained with it.

Clipping belongs to the trainer. Whenever a positive `gradient_clip_val` is
set, `d2l.Trainer` (that section) clips the gradients after
it has computed them and before the parameter update.

The method below returns the (possibly rescaled) gradients, and the compiled
training step hands them to the optimizer.

In [ ]:
@d2l.add_to_class(d2l.Trainer)
def clip_gradients(self, grad_clip_val, grads):
    grad_clip_val = tf.constant(grad_clip_val, dtype=tf.float32)
    new_grads = [tf.convert_to_tensor(grad) if isinstance(
        grad, tf.IndexedSlices) else grad for grad in grads]
    norm = tf.math.sqrt(sum((tf.reduce_sum(grad ** 2)) for grad in new_grads))
    scale = tf.minimum(1.0, grad_clip_val / norm)
    return [grad * scale for grad in new_grads]

## Training

The `d2l.TimeMachine` pipeline of that section downloads the
corpus, trains the BPE tokenizer on it, and serves minibatches of token-id
windows together with their shift-by-one targets. The corpus is 65,741 BPE
tokens long, and training window $i$ starts at token $i$. The pipeline
drops the 32 windows that straddle the split, so that no validation token
is ever a training target. By default the pipeline trains on the first
50,000 windows, most of the novella, and validates on the following 5,000.

In [ ]:
data = d2l.TimeMachine(batch_size=1024, num_steps=32)

Every window is $T = 32$ tokens long, and every forward pass starts from a
zero state. Neither information nor gradient crosses a window boundary, so
gradients flow backward through at most 32 steps: the chain of dependencies
is *truncated*. Whatever the novella does over longer ranges, the model can
learn it only insofar as it helps within a window. that section
examines what this compromise gives up, and what changes when the state is
carried from one window into the next.

### A Plain Model

We first train the model as built so far, the one whose parameters
that section counted: an embedding of dimension $d = 64$ and a cell
with $h = 128$ hidden units, for ten epochs of minibatch SGD at learning
rate 4, clipping gradients to norm $\theta = 1$, with no dropout. A
learning rate of 4 is large by the standards of the earlier chapters. Note
what it means together with clipping: whenever clipping acts, the update
has length $\eta\theta = 4$, whatever the gradient was. As in
that section, training is teacher-forced, so every step conditions on
the true prefix from the corpus. We keep a copy of the embedding table, to
see later how much training changes it.

In [ ]:
with d2l.try_gpu():
    rnn = RNNScratch(num_inputs=64, num_hiddens=128)
    model = RNNLMScratch(rnn, vocab_size=len(data.vocab), lr=4)

In [ ]:
E_init = d2l.numpy(model.E).copy()

In [ ]:
trainer = d2l.Trainer(max_epochs=10, gradient_clip_val=1, num_gpus=1)
ppl_init = math.exp(trainer.evaluate(model, data)['loss'])
print(f'initial validation perplexity {ppl_init:.0f}')
trainer.fit(model, data)

Before training, the validation perplexity is close to the vocabulary size
of 1,027, as predicted for a model that knows nothing yet. The board plots
the mean cross-entropy per token, the logarithm of the perplexity. Both
curves fall fast during the first epochs. Then they separate: the training
loss keeps falling, while the validation loss stops improving over the last
few epochs. After ten epochs the training perplexity is less than half the
validation perplexity. This should not surprise us. With 222,915 parameters
for 50,000 training tokens, more than four parameters per token, the model
has started to memorize its training text. The final validation perplexity
is the exponential of the last validation loss.

In [ ]:
ppl_plain = math.exp(trainer.history['val_loss'][-1])
print(f'validation perplexity {ppl_plain:.1f}')

How much of this result do we owe to clipping? We called it a crude
device, so let us see what happens without it. We train a second copy of
the plain model with the same settings, except that the gradients are not
clipped, and print the highest validation perplexity of the run next to the
final one.

In [ ]:
with d2l.try_gpu():
    unclipped = RNNLMScratch(RNNScratch(num_inputs=64, num_hiddens=128),
                             vocab_size=len(data.vocab), lr=4)

In [ ]:
trainer_unclipped = d2l.Trainer(max_epochs=10, num_gpus=1)  # No clipping
trainer_unclipped.fit(unclipped, data)
ppls = [math.exp(loss) for loss in trainer_unclipped.history['val_loss']]
print(f'validation perplexity without clipping: highest {max(ppls):.0f}, '
      f'final {ppls[-1]:.0f}')

Without clipping, training is erratic. At some epoch the loss jumps
instead of falling, as the argument for clipping warned, and the validation
perplexity leaps into the hundreds or thousands, in some runs into the
millions. Some runs recover part of the way and an occasional one never
does, but every run ends at more than twice the perplexity of the clipped
model, and most end far higher. Crude or not, clipping is what makes a
learning rate of 4 usable.

A perplexity of about 90 to 100 may look poor next to the perplexity of
about 6
of the character-level trigram in that section. The two
numbers cannot be compared: our model chooses among 1,027 BPE tokens per
step, a character model among 27 characters. Bits per byte
(that section, the equation) removes the dependence on
the tokenizer: it divides the total surprisal of a text by its length in
bytes, which here comes to $\log_2 \textrm{ppl}$ divided by the average
number of bytes per token. We measure the bytes per token on the 5,000
consecutive validation tokens that start the validation windows.

In [ ]:
ids = d2l.numpy(data.X[data.num_train:data.num_train+data.num_val, 0]).tolist()
bytes_per_token = len(data.tokenizer.decode(ids).encode('utf-8')) / len(ids)
print(f'{bytes_per_token:.2f} bytes/token, '
      f'{math.log2(ppl_plain) / bytes_per_token:.2f} bits per byte')

Each token covers about 2.8 bytes of text, and the plain model needs about
2.3 to 2.4 bits per byte.

### Embeddings under SGD

Overfitting is one problem of the plain model. A second one is hidden in
the embedding table, and it takes a measurement to find it: how far did
each row of $\mathbf{E}$ move during training, relative to its initial
length?

In [ ]:
E_final = d2l.numpy(model.E)

In [ ]:
change = (((E_final - E_init) ** 2).sum(1) / (E_init ** 2).sum(1)) ** 0.5
print(f'mean change of an embedding row: {change.mean():.1%} of its length')

The rows barely move. Each row receives a gradient only from the positions
where its token occurs, and the loss averages over all $1{,}024 \times 32 =
32{,}768$ positions of a minibatch, so a token that occurs a few times
contributes a few parts in 32,768 of the gradient. A step of SGD moves
every parameter by the learning rate times its gradient, the same rate for
all, which is far too small for rows that start at unit scale. Rows of
tokens that never occur in the training text do not move at all. This is an
awkward finding. The table that was meant to let the model choose a vector
for each token is, in this configuration, little more than a fixed random
projection.

We do not cure this here. We work around it. The input weights
$\mathbf{W}_{\textrm{xh}}$ do learn, so a wider embedding helps even if
its rows stay close to random: with $d = 512$ instead of 64,
$\mathbf{W}_{\textrm{xh}}$ has more random features of each token to
combine. Dropout (that section), applied to the embeddings at rate
0.1 and to the hidden states before the output layer at rate 0.5, counters
the overfitting.

### A Wider Embedding with Dropout

This is where the two dropout rates of `RNNLMScratch` come into use. The
new model has 740,355 parameters, most of them in the
$1{,}027 \times 512$ embedding table; everything else in the training
procedure stays the same.

In [ ]:
with d2l.try_gpu():
    rnn = RNNScratch(num_inputs=512, num_hiddens=128)
    model = RNNLMScratch(rnn, vocab_size=len(data.vocab), lr=4, dropout=0.5,
                         emb_dropout=0.1)

In [ ]:
trainer = d2l.Trainer(max_epochs=10, gradient_clip_val=1, num_gpus=1)
trainer.fit(model, data)

In [ ]:
ppl_scratch = math.exp(trainer.history['val_loss'][-1])
print(f'validation perplexity {ppl_scratch:.1f}, '
      f'{math.log2(ppl_scratch) / bytes_per_token:.2f} bits per byte')

The validation perplexity falls to about 60, against about 90 to 100 for
the plain model; in bits per byte, about 2.1 against 2.3 to 2.4. The training
loss on the board is measured with dropout active, which raises it, so it
no longer shows the gap between training and validation directly. The
validation loss is still at or near its lowest after ten epochs.

### Comparison with Kneser–Ney

Is this any good? The fair opponent is the best count-based model of
that section, interpolated Kneser–Ney smoothing
(that section). `d2l.kneser_ney` counts the $n$-grams of a
training sequence and returns the natural log-probability of every token of
another sequence, given the tokens before it. Both models must see the same
text, so we count the $n$-grams of the RNN's 50,000 training tokens, in the
same BPE tokenization. We score the validation tokens from the 33rd on,
`val[32:]`, so that the RNN, below, can read a full window of 32 validation
tokens before each of them.

In [ ]:
first_tokens = d2l.numpy(data.X[:, 0]).tolist()  # First token of each window
train = first_tokens[:data.num_train]
val = first_tokens[data.num_train:data.num_train + data.num_val]
lp_kns = {n: d2l.kneser_ney(train, val, len(data.vocab), n)[32:]
          for n in (2, 3, 4)}
for n, lp in lp_kns.items():
    bits = -sum(lp) / len(lp) / math.log(2)
    print(f'KN {n}-gram: {bits / bytes_per_token:.2f} bits per byte')
lp_kn = lp_kns[3]

The trigram is much better than the bigram, and a fourth token of context
helps it little more. We now score the same tokens `val[32:]` with the RNN,
each as the last target of a validation window, so that the RNN has read
the 32 tokens before it: validation window $j$ starts at `val[j]`, and its
last target is `val[j + 32]`. This treats the RNN more kindly than the
validation loss on the board, which averages over all 32 positions of a
window, the first of which see only one or a few tokens of context. The
cell computes the log-probability of every target of every window; we use
the last column now and the others shortly.

In [ ]:
start, stop = data.num_train, data.num_train + data.num_val - 32  # val[32:]

In [ ]:
X_val, Y_val = data.X[start:stop], data.Y[start:stop]
logp = d2l.numpy(tf.concat([  # Log-probability of every target
    tf.gather(tf.nn.log_softmax(model(X_val[i:i + 1024], training=False)),
              Y_val[i:i + 1024], batch_dims=2)
    for i in range(0, len(X_val), 1024)], 0))

Before we look at the result, note that the two models have very
different strengths. The RNN sees 32 tokens of context but learns only from
what it has seen in a few hundred SGD steps. The counts see two tokens but
remember every trigram of the training text. It is therefore natural to
combine them.
Linear interpolation, $\lambda P_\textrm{RNN} + (1 - \lambda)
P_\textrm{KN}$, is how @Mikolov.Kombrink.Burget.ea.2011 combined
their RNN language models with Kneser–Ney models, choosing $\lambda$ on
held-out data. We fix $\lambda = 0.5$ instead of tuning it, close to the
best weight for this pair. A weighted average of two probability
distributions is a probability distribution, so the mixture is a language
model too.

In [ ]:
lp_rnn = list(logp[:, -1])  # The last target of each window: val[32:]
lam = 0.5
lp_mix = [math.log(lam * math.exp(a) + (1 - lam) * math.exp(b))
          for a, b in zip(lp_rnn, lp_kn)]
for name, lp in [('RNN', lp_rnn), ('KN trigram', lp_kn), ('mixture', lp_mix)]:
    bits = -sum(lp) / len(lp) / math.log(2)
    print(f'{name}: {bits / bytes_per_token:.2f} bits per byte')
better = np.mean(np.array(lp_rnn) > np.array(lp_kn))
print(f'the RNN is the more confident model on {better:.0%} of the tokens')

After all this effort, the RNN alone does not beat a trigram. It comes
within a few hundredths of a bit per byte of Kneser–Ney, on the wrong side.
(Both numbers are higher than those in the table of
that section, which was computed on lowercased text with
letters only. Bits per byte compares tokenizers on the same text, and on
the raw text of this section the Kneser–Ney trigram needs about 2.1.) The
mixture is the more interesting number: it is better than the trigram by
about 0.06 bits per byte, which could not happen if the two models
assigned the same probabilities to the same tokens. Indeed, each is the
more confident of the two on about half of the tokens. The two models are
about equally good, and they are wrong in different places.

Why does a model that reads 32 tokens do no better than one that reads two?
Before we speculate, we can measure how much of its context the RNN uses.
In every validation window the first target is predicted from one token of
context, the second from two, and so on. Averaging the surprisal at each
position over all validation windows gives the loss as a function of the
context the model had, which we draw next to the Kneser–Ney trigram.

In [ ]:
bits = -logp.mean(0) / math.log(2) / bytes_per_token  # One per position
kn_bits = -sum(lp_kn) / len(lp_kn) / math.log(2) / bytes_per_token
positions = np.arange(1, data.num_steps + 1)
d2l.plot(positions, [bits, [kn_bits] * data.num_steps],
         'tokens of context', 'bits per byte', legend=['RNN', 'KN trigram'],
         figsize=(4.5, 3))
print('bits per byte with 1, 2, 3, 4, 8, 32 tokens of context:',
      [round(float(bits[k - 1]), 2) for k in (1, 2, 3, 4, 8, 32)])

The first target of a window, predicted from a single token, costs the
most, and the second and third much less. From about the fourth position on
the curve is flat: a model that has read thirty tokens predicts no better
than one that has read three. This answers the question. The trained RNN
uses only the last few tokens of its window, about as many as the trigram
sees, and so the two tie. Its state could carry information across all 32
steps. Training did not teach it to.

One cause of the shortfall we have already met: SGD left the embedding
table almost untrained. Adam (that section), an optimizer that adapts
the step to each parameter, moves the rows much further. Trained with it
and with slightly stronger embedding dropout, the RNN alone edges past the
trigram; an exercise asks you to try it.

## Text Generation

A trained language model can continue a prefix (that section). The
`predict` method below first *warms up* on the prefix, feeding its tokens
one at a time to build up a hidden state without emitting anything. It then
generates `num_tokens` new tokens, each time turning the current logits
$\mathbf{o}$ into a choice of next token and feeding that token back in as
the next input.

By default the choice is *greedy*: the method takes the highest-scoring
token at every step. With a positive `temperature` it samples from the
softmax of the scaled logits instead, a rule we leave to
that section.

In [ ]:
@d2l.add_to_class(RNNLMScratch)
def predict(self, prefix, num_tokens, tok, device=None, temperature=0.0):
    outputs, state = tok.encode(prefix), None
    for i in range(len(outputs) - 1):  # Warm up on the prefix
        X = d2l.tensor([[outputs[i]]])
        _, state = self.rnn(self.embedding(X), state)
    for _ in range(num_tokens):  # Generate num_tokens continuation tokens
        X = d2l.tensor([[outputs[-1]]])
        rnn_outputs, state = self.rnn(self.embedding(X), state)
        logits = self.output_layer(rnn_outputs)[0]  # Shape: (1, vocab_size)
        if temperature == 0:  # Greedy: the highest-scoring token
            token = tf.argmax(logits[0])
        else:  # Sample from the softmax of logits / temperature
            token = tf.random.categorical(logits / temperature, 1)[0, 0]
        outputs.append(int(token))
    return tok.decode(outputs)

We start with greedy decoding. The prefix is real text, not token ids: the
tokenizer trained by the pipeline (`data.tokenizer`) encodes it on the way
in and decodes the generated ids on the way out.

In [ ]:
d2l.print_wrapped(repr(model.predict('The Time Traveller', 50,
                                     data.tokenizer)))

The continuation has the look of English for a few words, and then it
often gets stuck. Greedy decoding is deterministic, and it tends to fall
into a loop: once the state returns to a phrase it has already produced,
the phrase comes back again and again. How to choose tokens better is the
subject of that section.

## Concise Implementation

Frameworks ship recurrent layers that bundle the cell and the loop over
time steps into one optimized call. The `RNN` class below wraps the
framework layer and keeps the interface of `RNNScratch`: it consumes a
(`num_steps`, `batch_size`, `num_inputs`) tensor and an optional state, and
returns the hidden states of all steps together with the final state. So
that the two implementations train the same model, `RNN` also initializes
the layer as `RNNScratch` does, with weights from $\mathcal{N}(0,
\sigma^2)$ and zero biases.

`SimpleRNN` takes batch-major inputs, so `forward` transposes on the way in
and out. As in `RNNScratch`, each weight gets its own initializer object.

In [ ]:
class RNN(d2l.Module):
    """The RNN model implemented with high-level APIs."""
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()
        normal = tf.keras.initializers.RandomNormal
        self.rnn = tf.keras.layers.SimpleRNN(
            num_hiddens, kernel_initializer=normal(stddev=sigma),
            recurrent_initializer=normal(stddev=sigma),
            return_sequences=True, return_state=True)

    # Keras builds the layer by calling `call` with named arguments, which
    # `Module.call(X, *args)` cannot pass on, so we name the state here
    def call(self, inputs, H=None, training=None):
        return self.forward(inputs, H)

    def forward(self, inputs, H=None):
        # The keras layer is batch-major: transpose in and out
        outputs, H = self.rnn(tf.transpose(inputs, perm=[1, 0, 2]), H)
        return tf.transpose(outputs, perm=[1, 0, 2]), H

Because the interface is unchanged, the language model carries over by
inheritance: `RNNLM` subclasses `RNNLMScratch` and swaps the hand-rolled
embedding table and output layer for the framework's embedding and dense
layers. Training, evaluation, and `predict` are all inherited. The new
layers are initialized like their scratch counterparts, with unit-normal
embedding rows, output weights from $\mathcal{N}(0, \sigma^2)$, and a zero
output bias.

In [ ]:
class RNNLM(d2l.RNNLMScratch):
    """The RNN-based language model implemented with high-level APIs."""
    def init_params(self):
        normal = tf.keras.initializers.RandomNormal
        self.emb = tf.keras.layers.Embedding(
            self.vocab_size, self.rnn.num_inputs,
            embeddings_initializer=normal(stddev=1))
        self.linear = tf.keras.layers.Dense(
            self.vocab_size, kernel_initializer=normal(stddev=self.sigma))

    def embedding(self, X):
        return self.emb(tf.transpose(X))

    def output_layer(self, hiddens):
        return d2l.transpose(self.linear(hiddens), (1, 0, 2))

We build the concise model with the settings of the wider scratch model.

In [ ]:
with d2l.try_gpu():
    rnn = RNN(num_inputs=512, num_hiddens=128)
    model = RNNLM(rnn, vocab_size=len(data.vocab), lr=4, dropout=0.5,
                  emb_dropout=0.1)

Training is the same call as before, with the same hyperparameters.

In [ ]:
trainer = d2l.Trainer(max_epochs=10, gradient_clip_val=1, num_gpus=1)
trainer.fit(model, data)

With the same initialization, data, and training, the concise model
reaches about the same validation perplexity as the scratch model, and its
greedy continuation gets stuck in the same way.

In [ ]:
ppl_concise = math.exp(trainer.history['val_loss'][-1])
pred = model.predict('The Time Traveller', 30, data.tokenizer)
d2l.print_wrapped(f'perplexity {ppl_concise:.1f}, {pred!r}')

The framework layer runs the same recurrence in a single optimized call, so
the rest of the book writes recurrent cells from scratch only to explain
them, and trains with the framework layers.

A tie with a trigram is less than the hidden state seemed to promise. The
loss by position gave the immediate reason: the trained model uses only the
last few tokens of its context. The deeper reason must lie in training. To
learn that a token thirty steps back matters, the model needs a gradient
that reaches thirty steps back, and that gradient passes through the same
transition at every step. that section computes it, and then
measures it in this very model.

## Summary

An RNN language model applies the same step at every position: it consumes
a token and the previous state and returns logits and a new state of the
same shape. The step is an embedding lookup, a tanh cell, and an output
layer, and each framework's recurrent layer computes the same model as our
tensor code. Training relies on two devices. Clipping the gradient norm
bounds the change any one update makes, and without it training at our
learning rate fails. Starting each window from a zero state truncates
backpropagation at the window length. On *The Time Machine*, with a wide
embedding and dropout, the RNN needs about 2.1 bits per byte, within a few
hundredths of a Kneser–Ney trigram on the same text, and the average of the
two models beats both. The loss by position shows why the RNN does no
better: it uses only the last few tokens of its 32-token window. Greedy
generation tends to fall into loops.

## Exercises

1. [code] **Gradient clipping ablation.** `d2l.Trainer` applies the
   projection the equation whenever `gradient_clip_val` is
   positive.
    1. Record the norm of the gradient before clipping at every update
       of the first epoch, and plot it. How often does clipping act, and
       when?
    1. The unclipped run of this section failed at `lr=4`. Halve the
       learning rate until three unclipped runs in a row train without a
       jump in the validation perplexity. How does their validation
       perplexity after ten epochs compare with that of the clipped run at
       `lr=4`? What, then, does clipping make possible?
    1. Replace $\tanh$ by ReLU in `RNNScratch` and train again, with and
       without clipping. Is clipping still needed? Explain the difference
       from the $\tanh$ cell in terms of the range of the hidden state.

1. [code] **Usable history.** Every training window starts from a zero
   state, so the model is never trained to use more than `num_steps`
   tokens of context, and the loss by position showed that it uses far
   fewer. `predict`, by contrast, carries the state across the whole
   prefix and every generated token. Evaluate the trained model
   on 512 consecutive validation tokens in two ways: resetting the state
   to zero every 32 tokens as in training, and carrying the state across
   all 512 tokens. Compare the average cross-entropy of the two. Does the
   carried state help, hurt, or make no difference, and what does the
   answer say about the states the model reaches beyond its training
   horizon?

1. [code] **Hyperparameters and data budget.**
    1. Starting from the model with the wider embedding, adjust the
       number of epochs, hidden units, embedding dimension, `num_steps`,
       learning rate, and dropout rates, or add weight decay, to lower the
       validation perplexity. Can the RNN alone, trained with SGD, beat
       the Kneser–Ney trigram? How does the relative improvement in bits
       per byte compare with the relative improvement in perplexity, and
       why?
    1. Set `num_train=10000` and train for 100
       epochs. Compare the training and validation perplexity curves and
       explain what you see. Consecutive windows overlap by 31 tokens, so
       how many parameters does the model have per distinct training
       token in this regime?

1. [code] **Adam.** Train the model with the wider embedding with the Adam
   optimizer of that section instead of SGD: subclass
   `RNNLMScratch`, override `configure_optimizers` to return Adam with
   learning rate 0.003, and keep clipping at 1.
    1. With `emb_dropout=0.2`, score the tokens `val[32:]` as in the
       comparison above, and compare its bits per byte with the SGD
       model, the Kneser–Ney trigram and 4-gram, and the mixture of the
       RNN and the trigram at $\lambda = 0.5$.
    1. Record the validation loss after every epoch, with
       `emb_dropout=0.1` and with 0.2. At which epoch is it lowest, and what
       happens after that?
    1. Measure again how far the embedding rows move during training. Why
       does Adam move them further than SGD does?

1. [code] **Vocabulary size and bits per byte.** ● Retrain the model with
   the wider embedding for `vocab_size` 512, 1,024, 2,048, and 4,096 in
   `d2l.TimeMachine`. The number of windows depends on the vocabulary size,
   so split by fraction: build each dataset once to read off
   `n = len(data.X)`, then build it again with `num_train=int(0.8 * n)` and
   `num_val=n - int(0.8 * n)`. (Setting `data.num_train` afterwards would
   skip the step that keeps validation tokens out of the training targets.)
   Every model then trains and validates on about the same stretch of text.
   Their per-token perplexities are not comparable, so compare them by bits
   per byte.
    1. Which vocabulary size gives the lowest bits per byte?
    1. How do the parameter counts of the embedding table and the output
       layer, and the length of the corpus in tokens, change with the
       vocabulary size? Which of these effects best explains the trend
       you measured?

1. [code] **Weight tying.** ● Weight tying sets the embedding dimension
   equal to the number of hidden units and uses $\mathbf{E}^\top$ as the
   output projection in place of $\mathbf{W}_{\textrm{hq}}$
   [@Press.Wolf.2017; @Inan.Khosravi.Socher.2017].
    1. How many parameters does tying save at $|\mathcal{V}| = 1{,}027$
       with 128 hidden units? GPT-2 small ties its $50{,}257 \times 768$
       token embedding to its output layer. How many parameters would
       untying add there, and what fraction of the model's roughly 124
       million parameters is that?
    1. Tie the weights in `RNNLMScratch` with an embedding dimension of 128.
       `RNNLMScratch` initializes $\mathbf{E}$ at unit scale
       and $\mathbf{W}_{\textrm{hq}}$ at scale `sigma`. What happens to
       the initial perplexity once the two are tied, and how do you
       restore a near-uniform initial prediction?
    1. Train the tied model and an untied model with a 128-dimensional
       embedding and 128 hidden units, and compare their validation
       perplexities.

1. [code] **Cross-book generalization.** ● Subclass `d2l.TimeMachine` and
   override `_download` to return the text of a different H. G. Wells
   novel, such as *The War of the Worlds* (Project Gutenberg ebook 36).
   Train the tokenizer and the model on it.
    1. Evaluate the model on held-out text of its own novel and on *The
       Time Machine*, encoding both with the new novel's tokenizer.
       Report perplexity and bits per byte for both.
    1. Write $s = \log_2(\textrm{ppl})$ for the per-token surprisal and
       $b$ for the bytes per token of an evaluated text, so that bits per
       byte is $s / b$. The ratio of bits per byte on *The Time Machine*
       (TM) to bits per byte on the held-out text of the new novel (own)
       factors as
       $$\frac{s_\textrm{TM}}{s_\textrm{own}} \cdot \frac{b_\textrm{own}}{b_\textrm{TM}}.$$
       The first factor reflects the model's predictions and the second
       the tokenizer's compression. Measure both factors. Which lies
       further from one?

[Discussions](https://d2l.discourse.group/t/1052)